# (Feature Engineering & Preprocessing)

In [1]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Ensure working directory is project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(f"Current Working Directory: {os.getcwd()}")

# Load Split Artifacts
train_df = pd.read_parquet('artifacts/data/03_train.parquet')
val_df = pd.read_parquet('artifacts/data/03_val.parquet')
test_df = pd.read_parquet('artifacts/data/03_test.parquet')

print(f" Loaded Splits successfully:")
print(f"  Train: {train_df.shape[0]:,} rows")
print(f"  Val:   {val_df.shape[0]:,} rows")
print(f"  Test:  {test_df.shape[0]:,} rows")


Current Working Directory: c:\Users\rosto\OneDrive\سطح المكتب\مشروع mlops
 Loaded Splits successfully:
  Train: 67,529 rows
  Val:   14,470 rows
  Test:  14,471 rows


## 1. (Inference Horizon & Anti-Leakage Audit)  
**(Future Columns):**  
`order_delivered_customer_date`  
`order_delivered_carrier_date`  
`review_score` , `review_count`  
`delivery_delay_days`  
**(Non-predictive Identifiers):**  
`order_id`, `customer_id`, `seller_id`, `customer_unique_id`  
**(Available at Order Time):**   
for exampel (`order_estimated_delivery_date`)

## 2. (Feature Engineering Logic)

In [2]:
# Brazilian macro-regions mapping
STATE_TO_REGION = {
    # Southeast
    'SP': 'Southeast', 'RJ': 'Southeast', 'MG': 'Southeast', 'ES': 'Southeast',
    # South
    'PR': 'South', 'SC': 'South', 'RS': 'South',
    # Northeast
    'BA': 'Northeast', 'PE': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast',
    'PB': 'Northeast', 'RN': 'Northeast', 'AL': 'Northeast', 'PI': 'Northeast', 'SE': 'Northeast',
    # North
    'AM': 'North', 'PA': 'North', 'RO': 'North', 'TO': 'North', 'AC': 'North', 'AP': 'North', 'RR': 'North',
    # Center-West
    'GO': 'Center-West', 'MT': 'Center-West', 'MS': 'Center-West', 'DF': 'Center-West'
}

def engineer_features(df):
    data = df.copy()
    
    # 1. Ensure datetime
    data['order_purchase_timestamp'] = pd.to_datetime(data['order_purchase_timestamp'])
    data['order_estimated_delivery_date'] = pd.to_datetime(data['order_estimated_delivery_date'])
    
    # 2. Promised lead time in days
    data['estimated_delivery_days'] = (
        data['order_estimated_delivery_date'] - data['order_purchase_timestamp']
    ).dt.total_seconds() / 86400.0
    
    # 3. Calendar & purchase time features
    data['purchase_month'] = data['order_purchase_timestamp'].dt.month
    data['purchase_dayofweek'] = data['order_purchase_timestamp'].dt.dayofweek
    data['purchase_hour'] = data['order_purchase_timestamp'].dt.hour
    data['purchase_is_weekend'] = data['purchase_dayofweek'].isin([5, 6]).astype(int)
    
    # 4. Freight and pricing ratios
    data['freight_ratio'] = data['total_freight'] / (data['total_price'] + 1.0)
    data['freight_per_item'] = data['total_freight'] / (data['order_items_count'] + 1e-5)
    data['price_per_item'] = data['total_price'] / (data['order_items_count'] + 1e-5)
    
    # 5. Density (weight / volume)
    data['density_g_cm3'] = data['total_weight_g'] / (data['total_volume_cm3'] + 1.0)
    
    # 6. Macro-regions & cross-region flags
    data['customer_region'] = data['customer_state'].map(STATE_TO_REGION).fillna('Other')
    data['seller_region'] = data['seller_state'].map(STATE_TO_REGION).fillna('Other')
    data['is_inter_region'] = (data['customer_region'] != data['seller_region']).astype(int)
    data['is_same_state'] = (data['customer_state'] == data['seller_state']).astype(int)
    
    return data

# Apply feature engineering to all splits
train_feat = engineer_features(train_df)
val_feat = engineer_features(val_df)
test_feat = engineer_features(test_df)

print(f" Feature engineering applied across all splits. Sample generated features:")
train_feat[['estimated_delivery_days', 'freight_ratio', 'customer_region', 'is_inter_region', 'purchase_is_weekend']].head()


 Feature engineering applied across all splits. Sample generated features:


,estimated_delivery_days,freight_ratio,customer_region,is_inter_region,purchase_is_weekend
0,18.488449,0.062440,Southeast,1,0
1,23.593866,0.503560,Southeast,1,0
2,34.293866,0.750655,South,1,0
3,56.115556,0.459856,South,1,0
4,50.079132,0.112159,Southeast,0,0


## 3. (Scikit-Learn ColumnTransformer)  
**(Numerical Features):**  
(`SimpleImputer(strategy='median')`)  
(`StandardScaler()`)  
**(Categorical Features):**  
processing `missing` __ (`OneHotEncoder(handle_unknown='ignore')`)


In [3]:
# Define feature subsets
num_features = [
    'order_items_count', 'total_price', 'total_freight', 'total_weight_g',
    'total_volume_cm3', 'total_payment_value', 'max_payment_installments',
    'distance_km', 'estimated_delivery_days', 'purchase_month',
    'purchase_dayofweek', 'purchase_hour', 'freight_ratio',
    'freight_per_item', 'price_per_item', 'density_g_cm3']

cat_features = [
    'customer_state', 'seller_state', 'dominant_payment_type',
    'customer_region', 'seller_region', 'is_same_state',
    'is_inter_region', 'purchase_is_weekend']

# Separate features (X) and target (y)
X_train_raw = train_feat[num_features + cat_features]
y_train = train_feat['is_late']

X_val_raw = val_feat[num_features + cat_features]
y_val = val_feat['is_late']

X_test_raw = test_feat[num_features + cat_features]
y_test = test_feat['is_late']

# Build Pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
], remainder='drop')

print(" Preprocessor pipeline designed successfully:")
print(f"  Numerical Features ({len(num_features)}):", num_features)
print(f"  Categorical Features ({len(cat_features)}):", cat_features)


 Preprocessor pipeline designed successfully:
  Numerical Features (16): ['order_items_count', 'total_price', 'total_freight', 'total_weight_g', 'total_volume_cm3', 'total_payment_value', 'max_payment_installments', 'distance_km', 'estimated_delivery_days', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'freight_ratio', 'freight_per_item', 'price_per_item', 'density_g_cm3']
  Categorical Features (8): ['customer_state', 'seller_state', 'dominant_payment_type', 'customer_region', 'seller_region', 'is_same_state', 'is_inter_region', 'purchase_is_weekend']


## 4. (Fit on Train ONLY)

In [4]:
# 1. Fit ONLY on training data
preprocessor.fit(X_train_raw)

# 2. Extract generated feature names
cat_encoded_names = list(preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_features))
all_feature_names = num_features + cat_encoded_names

print(f" Total Output Features: {len(all_feature_names)}")

# 3. Transform all splits
X_train_transformed = preprocessor.transform(X_train_raw)
X_val_transformed = preprocessor.transform(X_val_raw)
X_test_transformed = preprocessor.transform(X_test_raw)

# 4. Convert back to DataFrames with feature names and attach target
train_features_df = pd.DataFrame(X_train_transformed, columns=all_feature_names)
train_features_df['is_late'] = y_train.values

val_features_df = pd.DataFrame(X_val_transformed, columns=all_feature_names)
val_features_df['is_late'] = y_val.values

test_features_df = pd.DataFrame(X_test_transformed, columns=all_feature_names)
test_features_df['is_late'] = y_test.values

print(f"Processed Shapes:")
print(f"  Train: {train_features_df.shape}")
print(f"  Val:   {val_features_df.shape}")
print(f"  Test:  {test_features_df.shape}")


 Total Output Features: 86
Processed Shapes:
  Train: (67529, 87)
  Val:   (14470, 87)
  Test:  (14471, 87)


## 5. (Sanity Checks)

In [5]:
# Check for any remaining nulls
assert train_features_df.isnull().sum().sum() == 0, " Null values found in processed train features!"
assert val_features_df.isnull().sum().sum() == 0, " Null values found in processed val features!"
assert test_features_df.isnull().sum().sum() == 0, " Null values found in processed test features!"

print(" PASSED: Zero missing values across all feature sets!")
print(f"Target distribution in Train features: {train_features_df['is_late'].value_counts(normalize=True).to_dict()}")
print(f"Target distribution in Val features:   {val_features_df['is_late'].value_counts(normalize=True).to_dict()}")
print(f"Target distribution in Test features:  {test_features_df['is_late'].value_counts(normalize=True).to_dict()}")


 PASSED: Zero missing values across all feature sets!
Target distribution in Train features: {0: 0.9097276725554947, 1: 0.09027232744450532}
Target distribution in Val features:   {0: 0.9465791292328957, 1: 0.053420870767104355}
Target distribution in Test features:  {0: 0.9338677354709419, 1: 0.06613226452905811}


## 6.(Save Artifacts)

In [6]:
# Save feature parquet files
train_features_df.to_parquet('artifacts/data/05_train_features.parquet', index=False)
val_features_df.to_parquet('artifacts/data/05_val_features.parquet', index=False)
test_features_df.to_parquet('artifacts/data/05_test_features.parquet', index=False)

# Save preprocessor transformer object
joblib.dump(preprocessor, 'artifacts/models/preprocessor.joblib')

# Save feature names list
with open('artifacts/models/feature_names.json', 'w', encoding='utf-8') as f:
    json.dump(all_feature_names, f, indent=2)

print(" Artifacts saved successfully:")

 Artifacts saved successfully:
